# Python Bug Detection Using CodeBERT on the BugInPy Dataset

**PhD topic:** Large Language Models for Automated Program Debugging
**Experiment 1:** Binary classification of Python code units — `1 = buggy`, `0 = non-buggy` — by fine-tuning `microsoft/codebert-base` on BugInPy.

---

## How to use this notebook

* This notebook is designed for **Google Colab** (`Runtime → Change runtime type → GPU`, e.g. T4).
* Run it **cell by cell**. Do **not** use `Runtime → Run all`.
* Every section follows the same structure:
  1. **Explanation** — what the step does.
  2. **Scientific objective** — why the step matters for the validity of the experiment.
  3. **Code** — the cell(s) to execute.
  4. **Expected output** — what you should see.
  5. **Verification** — an automated check (`assert`) that fails loudly if something is wrong.
  6. **Interpretation** — how to read the result.
* No results in this notebook are pre-filled. All numbers come from your own execution.

## Experimental phases

| Phase | Title | Status |
|---|---|---|
| 1 | Environment setup | done |
| 2 | BugInPy download | **available** |
| 3 | Dataset exploration | pending |
| 4 | Dataset construction (labeling strategy) | pending |
| 5 | Data preprocessing & tokenization | pending |
| 6 | Leakage-free dataset splitting | pending |
| 7 | Baselines (TF-IDF + LR / RF) | pending |
| 8 | CodeBERT fine-tuning | pending |
| 9 | Evaluation | pending |
| 10 | Scientific analysis & threats to validity | pending |

Phases are added to this notebook incrementally, after the results of the previous phase are confirmed.

---
# PHASE 1 — Environment Setup

**Goal of the phase:** build a known, documented and reproducible computational environment before touching any data.

Sections:
* 1.1 Python version and global reproducibility flags
* 1.2 GPU availability and hardware information
* 1.3 Installation of required libraries
* 1.4 Imports and library versions
* 1.5 Random seeds and determinism
* 1.6 (Optional) Google Drive mounting and experiment directory layout
* 1.7 Experiment configuration and environment record
* 1.8 Phase 1 verification summary

## 1.1 Python version and global reproducibility flags

**Explanation.** We print the Python interpreter version and platform, and set environment variables that must exist *before* PyTorch initialises CUDA.

**Scientific objective.** The software stack is part of the experimental setup. Recording it allows other researchers (and reviewers) to reproduce the run. `CUBLAS_WORKSPACE_CONFIG` is required by PyTorch for deterministic cuBLAS operations; `PYTHONHASHSEED` fixes hashing in this process (e.g. ordering of `set` iteration in child processes).

**Expected output.** A Python version ≥ 3.9 (Colab currently ships Python 3.10–3.12), the platform string, and confirmation that the variables are set.

In [ ]:
# ---- 1.1 Python version & reproducibility flags ----
import os, sys, platform

# Must be set BEFORE torch initialises CUDA (i.e. before the first `import torch`).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # avoids fork warnings from HF tokenizers

IN_COLAB = "google.colab" in sys.modules

print("Python version :", sys.version.split()[0])
print("Implementation :", platform.python_implementation())
print("Platform       :", platform.platform())
print("Running in Colab:", IN_COLAB)
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

# ---- Verification ----
assert sys.version_info >= (3, 9), "Python >= 3.9 is required."
if not IN_COLAB:
    print("WARNING: this notebook is designed for Google Colab; some cells (Drive) will be skipped.")
print("\n[OK] 1.1 passed")

**Interpretation.** If `Running in Colab` is `False`, you are not in Colab (e.g. local Jupyter); the notebook still works but Drive integration is skipped. Note the Python version — it will be stored automatically in the environment record (1.7).

## 1.2 GPU availability and hardware information

**Explanation.** We query the NVIDIA driver (`nvidia-smi`) and PyTorch's view of CUDA, and record CPU / RAM / disk.

**Scientific objective.** Training time, maximum batch size and even small numerical differences depend on the hardware. Fine-tuning CodeBERT (≈125 M parameters) on CPU is impractically slow, so a GPU is strongly recommended for Phase 8. Baselines (Phase 7) can run on CPU.

**Expected output.**
* With a GPU runtime: an `nvidia-smi` table (typically *Tesla T4, 15 GB*; on Colab Pro possibly L4 / A100), `CUDA available: True`, the GPU name and memory.
* Without a GPU: `CUDA available: False` and a warning. In that case go to **Runtime → Change runtime type → T4 GPU**, then restart from cell 1.1.

In [ ]:
# ---- 1.2 GPU & hardware information ----
import subprocess, shutil

def run(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return f"<failed: {e}>"

print("=== nvidia-smi ===")
print(run("nvidia-smi") or "nvidia-smi not available (no GPU attached).")

import torch
cuda_ok = torch.cuda.is_available()
print("\n=== PyTorch view ===")
print("torch version   :", torch.__version__)
print("CUDA available  :", cuda_ok)
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print("CUDA (torch)    :", torch.version.cuda)
    print("cuDNN           :", torch.backends.cudnn.version())
    print("GPU count       :", torch.cuda.device_count())
    print("GPU name        :", props.name)
    print(f"GPU memory      : {props.total_memory / 1024**3:.1f} GB")
    print(f"Compute capab.  : {props.major}.{props.minor}")
    print("bf16 supported  :", torch.cuda.is_bf16_supported())

print("\n=== CPU / RAM / Disk ===")
print("CPU             :", run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'") or platform.processor())
print("Logical cores   :", os.cpu_count())
print("RAM             :", run("free -h | awk '/Mem:/ {print $2\" total, \"$7\" available\"}'"))
total, used, free = shutil.disk_usage("/")
print(f"Disk (/)        : {total/1024**3:.0f} GB total, {free/1024**3:.0f} GB free")

DEVICE = torch.device("cuda" if cuda_ok else "cpu")
print("\nSelected DEVICE :", DEVICE)

# ---- Verification ----
if not cuda_ok:
    print("\nWARNING: No GPU detected. Phases 1-7 can run on CPU, but Phase 8 (CodeBERT) "
          "will be extremely slow. Switch to a GPU runtime before Phase 8.")
else:
    x = torch.randn(1024, 1024, device=DEVICE)
    assert torch.isfinite(x @ x).all(), "GPU matmul sanity check failed."
    print("GPU matmul sanity check: OK")
print("\n[OK] 1.2 passed")

**Interpretation.** Note the GPU model and memory: with a 15 GB T4, CodeBERT with `max_length=512` fits comfortably with batch size 8–16 (mixed precision). The GPU name is stored in the environment record, and will be reported together with training time in Phase 8.

## 1.3 Installation of required libraries

**Explanation.** Colab already provides `torch`, `numpy`, `pandas`, `scikit-learn` and `matplotlib`. We install/upgrade the Hugging Face stack and a few utilities. We deliberately **do not reinstall PyTorch**, to keep the CUDA build that matches Colab's driver.

| Library | Role in the experiment |
|---|---|
| `transformers` | CodeBERT model + tokenizer, training utilities |
| `datasets` | efficient tokenized dataset storage |
| `accelerate` | required by `transformers` Trainer / mixed precision |
| `evaluate` | standard metric implementations (cross-check) |
| `scikit-learn` | TF-IDF baselines, metrics, splitting utilities |
| `seaborn`, `matplotlib` | figures |
| `tqdm` | progress bars |
| `GitPython` | inspecting BugInPy git metadata (Phase 2–4) |

**Scientific objective.** A controlled and recorded dependency set. Exact installed versions are logged in 1.4 and saved in 1.7.

**Expected output.** pip progress (quiet mode prints little). Possible *dependency-resolver warnings* about unrelated Colab packages are normally harmless. If pip reports that a **restart is required**, use `Runtime → Restart session`, then re-run from **1.1** (skip 1.3).

In [ ]:
# ---- 1.3 Install libraries (run once per Colab session) ----
%pip install -q -U "transformers>=4.44,<5" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" \
                   "scikit-learn>=1.3" seaborn tqdm GitPython
print("\n[OK] 1.3 installation finished — check above for errors (warnings are usually fine).")

**Verification.** The next cell (1.4) imports every library and prints its version; an `ImportError` there means an installation failed.

**Interpretation.** Version ranges (not exact pins) are used because Colab images change over time; the *exact* versions actually used are what matters for reproducibility, and those are recorded.

## 1.4 Imports and library versions

**Explanation.** Import every dependency used across all phases and print the versions.

**Scientific objective.** Fail early if anything is missing, and document the precise software versions.

**Expected output.** A table of library versions, e.g. `transformers 4.4x.x`, `torch 2.x.x+cu12x`, and `[OK] 1.4 passed`.

In [ ]:
# ---- 1.4 Imports ----
import json, time, random, re, ast, hashlib, textwrap, warnings, datetime, logging
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import sklearn
import transformers
import datasets
import accelerate
import evaluate
import git  # GitPython

from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)

LIB_VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "evaluate": evaluate.__version__,
    "scikit-learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "GitPython": git.__version__,
}
print(pd.Series(LIB_VERSIONS, name="version").to_string())

# ---- Verification ----
assert int(transformers.__version__.split(".")[0]) == 4, "Expected transformers 4.x"
print("\n[OK] 1.4 passed")

**Interpretation.** These versions are saved to `environment.json` in 1.7 and must be reported in the methodology section of the paper/thesis.

## 1.5 Random seeds and determinism

**Explanation.** Fix the seeds of Python `random`, NumPy and PyTorch (CPU + CUDA), and enable deterministic algorithms.

**Scientific objective.** Fine-tuning results vary with weight initialisation of the classification head, data shuffling and dropout. Fixing seeds makes a single run repeatable. *Note:* full bit-level determinism on GPU is not always guaranteed (some CUDA kernels are non-deterministic); for the final thesis results we will therefore also report the **mean ± std over several seeds** (planned in Phase 8/9).

**Expected output.** `Seed set to 42`, followed by two identical random draws (re-seeding test) and `[OK] 1.5 passed`.

In [ ]:
# ---- 1.5 Seeds ----
SEED = 42

def set_seed(seed: int = SEED, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # warn_only=True: report (not crash on) ops without a deterministic implementation
        torch.use_deterministic_algorithms(True, warn_only=True)
    print(f"Seed set to {seed} (deterministic={deterministic})")

set_seed(SEED)

# ---- Verification: re-seeding must reproduce the same numbers ----
def draw():
    return (random.random(), float(np.random.rand()), float(torch.rand(1)))
set_seed(SEED); a = draw()
set_seed(SEED); b = draw()
print("draw 1:", a)
print("draw 2:", b)
assert a == b, "Seeding is not reproducible!"
set_seed(SEED)
print("\n[OK] 1.5 passed")

**Interpretation.** Identical draws confirm that every random generator used by the pipeline is controlled by `SEED`. The seed value is recorded in the configuration (1.7).

## 1.6 (Optional) Google Drive mounting and directory layout

**Explanation.** Colab's local disk is erased when the session ends. Mounting Google Drive lets us persist the processed dataset, models, checkpoints, logs, metrics and figures. Set `USE_DRIVE = False` to work only on the temporary local disk.

When the cell runs, Colab will open an authorisation pop-up — accept access to your Drive.

**Directory layout** (created under `MyDrive/PhD_Experiments/BugInPy_CodeBERT/`):

```
BugInPy_CodeBERT/
├── data/
│   ├── raw/          # untouched copy of BugInPy (Phase 2) — never modified
│   ├── interim/      # extracted code units before cleaning (Phase 4)
│   └── processed/    # final labelled dataset + split files (Phases 5-6)
├── models/
│   ├── baselines/    # TF-IDF + LR / RF (Phase 7)
│   └── codebert/     # final fine-tuned model + tokenizer (Phase 8)
├── checkpoints/      # intermediate training checkpoints
├── logs/             # training logs, environment & config records
├── results/
│   ├── metrics/      # CSV / JSON evaluation results
│   └── predictions/  # per-sample predictions (error analysis)
└── figures/          # plots (loss curves, ROC, confusion matrix ...)
```

**Scientific objective.** Persistent, organised storage of every artefact, which is required for reproducibility and for later phases of the PhD.

**Expected output.** `Mounted at /content/drive` (if `USE_DRIVE=True`) and a listing of the created folders.

In [ ]:
# ---- 1.6 Optional Google Drive + directory layout ----
USE_DRIVE = True          # <-- set to False to keep everything on the temporary Colab disk
EXPERIMENT_NAME = "BugInPy_CodeBERT"

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/PhD_Experiments") / EXPERIMENT_NAME
else:
    if USE_DRIVE and not IN_COLAB:
        print("Not running in Colab: Drive unavailable, using local directory.")
    BASE_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / EXPERIMENT_NAME

# Fast local working dir (git clone etc. is much faster on local disk than on Drive)
WORK_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / "work"

PATHS = {
    "base":        BASE_DIR,
    "raw":         BASE_DIR / "data" / "raw",
    "interim":     BASE_DIR / "data" / "interim",
    "processed":   BASE_DIR / "data" / "processed",
    "baselines":   BASE_DIR / "models" / "baselines",
    "codebert":    BASE_DIR / "models" / "codebert",
    "checkpoints": BASE_DIR / "checkpoints",
    "logs":        BASE_DIR / "logs",
    "metrics":     BASE_DIR / "results" / "metrics",
    "predictions": BASE_DIR / "results" / "predictions",
    "figures":     BASE_DIR / "figures",
    "work":        WORK_DIR,
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    print(f"{k:<12} -> {p}")

# ---- Verification: every directory exists and is writable ----
for k, p in PATHS.items():
    assert p.is_dir(), f"Missing dir: {p}"
    probe = p / ".write_test"
    probe.write_text("ok"); probe.unlink()
print("\nStorage persistent across sessions:", USE_DRIVE and IN_COLAB)
print("[OK] 1.6 passed")

**Interpretation.** If `Storage persistent across sessions: False`, all outputs will be lost when the Colab runtime is recycled — acceptable for testing, not for final experiments. `WORK_DIR` (local disk) is used for heavy temporary operations; final artefacts are always copied to `BASE_DIR`.

## 1.7 Experiment configuration and environment record

**Explanation.** A single `CONFIG` dictionary centralises every experimental choice (seed, model, hyperparameters, split ratios). Later phases read from it — nothing is hard-coded elsewhere. The configuration and full environment description are saved as JSON with a unique **run ID**.

**Scientific objective.** Research reproducibility: seed, dataset version, model name, hyperparameters and hardware are all recorded. Fields marked `None` are filled in by later phases (e.g. the exact BugInPy commit hash in Phase 2, split identifiers in Phase 6, training time in Phase 8).

**Expected output.** The printed configuration and the paths of `config_<RUN_ID>.json` and `environment_<RUN_ID>.json` in `logs/`.

In [ ]:
# ---- 1.7 Configuration & environment record ----
# After a Colab restart, put your previous RUN_ID here to continue the SAME run
# (e.g. RESUME_RUN_ID = "20261112_122126"). Leave None to start a new run.
RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    "run_id": RUN_ID,
    "experiment": EXPERIMENT_NAME,
    "seed": SEED,
    # --- dataset (filled in Phase 2) ---
    "dataset": {
        "name": "BugInPy",
        "repository": "https://github.com/soarsmu/BugsInPy",
        "commit": None,          # exact commit hash recorded in Phase 2
    },
    # --- model ---
    "model_name": "microsoft/codebert-base",
    "num_labels": 2,
    "label_map": {"0": "non-buggy", "1": "buggy"},
    # --- training hyperparameters (Phase 8) ---
    "hyperparameters": {
        "optimizer": "AdamW",
        "learning_rate": 2e-5,
        "batch_size": 16 if cuda_ok else 8,
        "num_epochs": 5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "max_seq_length": 512,
        "early_stopping_patience": 2,
        "early_stopping_metric": "eval_f1",
        "loss": "CrossEntropyLoss",
        "fp16": bool(cuda_ok),
    },
    # --- splitting (Phase 6) ---
    "split": {
        "strategy": "project-level (grouped)",
        "ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
        "split_ids_file": None,  # filled in Phase 6
    },
    "paths": {k: str(v) for k, v in PATHS.items()},
}

ENVIRONMENT = {
    "run_id": RUN_ID,
    "timestamp": datetime.datetime.now().isoformat(),
    "in_colab": IN_COLAB,
    "platform": platform.platform(),
    "libraries": LIB_VERSIONS,
    "hardware": {
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if cuda_ok else None,
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda_ok else None,
        "cuda_version": torch.version.cuda if cuda_ok else None,
        "cudnn_version": torch.backends.cudnn.version() if cuda_ok else None,
        "cpu_count": os.cpu_count(),
        "cpu_model": run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'"),
    },
    "pip_freeze_file": f"pip_freeze_{RUN_ID}.txt",
}

def save_json(obj, path):
    path = Path(path)
    path.write_text(json.dumps(obj, indent=2, default=str))
    return path

cfg_path = save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
env_path = save_json(ENVIRONMENT, PATHS["logs"] / f"environment_{RUN_ID}.json")
(PATHS["logs"] / ENVIRONMENT["pip_freeze_file"]).write_text(run(f"{sys.executable} -m pip freeze"))

print(json.dumps(CONFIG, indent=2)[:2500])
print("\nSaved:", cfg_path)
print("Saved:", env_path)
print("Saved:", PATHS["logs"] / ENVIRONMENT["pip_freeze_file"])

# ---- Verification: files can be read back and match ----
assert json.loads(cfg_path.read_text())["seed"] == SEED
assert json.loads(env_path.read_text())["libraries"]["transformers"] == transformers.__version__
print("\n[OK] 1.7 passed")

**Interpretation.** `RUN_ID` uniquely identifies this experimental run; every file produced later will carry it. The `pip_freeze` file is the most precise record of the software environment and can be used to rebuild it exactly.

> **Note on hyperparameters.** The values above follow your specification (AdamW, lr = 2e-5, batch 8/16, 5 epochs, early stopping, cross-entropy). `weight_decay=0.01`, `warmup_ratio=0.1` and `max_seq_length=512` are standard defaults for CodeBERT fine-tuning; they will be discussed again in Phase 5 (sequence length) and Phase 8 before training.

## 1.8 Phase 1 verification summary

**Explanation.** A final checklist of the whole phase.

**Expected output.** A table where every row says `PASS` (the GPU row may say `WARN` on a CPU runtime).

In [ ]:
# ---- 1.8 Phase 1 summary ----
checks = {
    "Python >= 3.9":                 "PASS" if sys.version_info >= (3, 9) else "FAIL",
    "GPU available":                 "PASS" if cuda_ok else "WARN (CPU only)",
    "transformers importable":       "PASS",
    "Seeds reproducible":            "PASS",
    "Output directories writable":   "PASS",
    "Persistent storage (Drive)":    "PASS" if (USE_DRIVE and IN_COLAB) else "WARN (temporary disk)",
    "Config + environment saved":    "PASS" if cfg_path.exists() and env_path.exists() else "FAIL",
}
summary = pd.DataFrame({"check": list(checks), "status": list(checks.values())})
print(summary.to_string(index=False))
print(f"\nRUN_ID = {RUN_ID}   DEVICE = {DEVICE}   BASE_DIR = {BASE_DIR}")
assert "FAIL" not in checks.values(), "Phase 1 has failing checks — fix them before continuing."
print("\n[OK] PHASE 1 COMPLETE — report these outputs before moving to Phase 2.")

**Interpretation.** Phase 1 is complete when no row is `FAIL`. Please report back:
1. The output of **1.2** (GPU name & memory),
2. The library versions from **1.4**,
3. The summary table of **1.8**,
4. Any warning or error message.

➡️ Continue with **PHASE 2** below.

---
# PHASE 2 — BugInPy Download

> **If your Colab session was restarted** (the variables are gone), first re-run Phase 1:
> 1.1 → 1.2 → 1.3 → 1.4 → 1.5 → 1.6 → 1.7 (set `RESUME_RUN_ID` to your previous run ID in 1.7) → then continue here.

**Goal of the phase:** obtain the official BugInPy benchmark, pin its exact version, keep an untouched archived copy, and verify that every file is accessible.

### Important scientific note: what BugInPy actually contains
BugInPy (official repository: [`soarsmu/BugsInPy`](https://github.com/soarsmu/BugsInPy), Widyasari et al., ESEC/FSE 2020) is a **benchmark of metadata**, not a source-code archive. For each project and each bug it stores:

| File | Content |
|---|---|
| `projects/<project>/project.info` | GitHub URL of the real project |
| `projects/<project>/bugs/<id>/bug.info` | `buggy_commit_id`, `fixed_commit_id`, `test_file`, `python_version` |
| `projects/<project>/bugs/<id>/bug_patch.txt` | the developer's fix as a unified *diff* (buggy → fixed) |
| `projects/<project>/bugs/<id>/run_test.sh` | the test command(s) that expose the bug |
| `requirements.txt`, `setup.sh` | environment needed to run the tests |

The actual **buggy and fixed source code** is obtained later (Phase 4) by checking out the two commits of each original project. Phase 2 therefore downloads only the benchmark itself (~13 MB).

Sections:
* 2.1 Clone the official repository and record its exact commit
* 2.2 Preserve the original dataset (archive + checksums)
* 2.3 Inspect the directory structure
* 2.4 Verify that all files are accessible and well-formed
* 2.5 Record the dataset version and Phase 2 summary

## 2.1 Clone the official BugInPy repository

**Explanation.** We `git clone` the official repository onto the fast local disk (`/content/work/BugsInPy`) and record the exact commit hash, date and message of the version obtained.

**Scientific objective.** The repository is still maintained, so "BugInPy" is not a fixed object: different download dates can give different bug sets. The commit hash is the **dataset version** that must be reported in the thesis. To reproduce a previous run exactly, set `BUGSINPY_COMMIT` to that hash.

**Expected output.** A commit hash (40 hex characters), its date, and `Working tree clean: True`. (When this notebook was prepared, the latest commit was `11c5f1ee…` from 9 Oct 2025. Yours may be newer if the repository has been updated since.)

In [ ]:
# ---- 2.1 Clone BugsInPy ----
BUGSINPY_URL    = CONFIG["dataset"]["repository"] + ".git"
BUGSINPY_COMMIT = None      # None = latest; or pin a hash, e.g. "11c5f1eea954a42132cfd06bf257766a7963e0fd"
BUGSINPY_DIR    = PATHS["work"] / "BugsInPy"

if not (BUGSINPY_DIR / ".git").exists():
    print(f"Cloning {BUGSINPY_URL} ...")
    t0 = time.time()
    git.Repo.clone_from(BUGSINPY_URL, BUGSINPY_DIR)
    print(f"Cloned in {time.time() - t0:.1f}s")
else:
    print("Repository already present locally — reusing it.")

bip_repo = git.Repo(BUGSINPY_DIR)
if BUGSINPY_COMMIT:
    bip_repo.git.checkout(BUGSINPY_COMMIT)

head = bip_repo.head.commit
BUGSINPY_VERSION = {
    "url": BUGSINPY_URL,
    "commit": head.hexsha,
    "commit_date": head.committed_datetime.isoformat(),
    "commit_message": head.message.strip().splitlines()[0],
    "downloaded_at": datetime.datetime.now().isoformat(),
}
for k, v in BUGSINPY_VERSION.items():
    print(f"{k:<15}: {v}")
print("Working tree clean:", not bip_repo.is_dirty(untracked_files=True))

# ---- Verification ----
assert re.fullmatch(r"[0-9a-f]{40}", head.hexsha)
assert (BUGSINPY_DIR / "projects").is_dir(), "'projects/' folder not found in the clone."
assert not bip_repo.is_dirty(untracked_files=True), "Local clone was modified!"
print("\n[OK] 2.1 passed")

**Interpretation.** The printed `commit` is the dataset version of this experiment; it is written into `CONFIG` in 2.5. A clean working tree means the local copy is identical to the official version.

## 2.2 Preserve the original dataset

**Explanation.** We store an untouched copy of the benchmark in `data/raw/` on Drive:
* `BugsInPy_<commit>.tar.gz`: a compressed archive of the full clone (including the git history);
* `BugsInPy_<commit>_manifest.csv`: path, size and **SHA-256** of every file;
* `BugsInPy_<commit>_source.json`: where and when the data came from, plus the archive checksum.

Existing archives are never overwritten. Later phases only **read** from the local clone, and the manifest lets us prove at any time that no original file was modified.

**Scientific objective.** Data provenance and integrity: the raw data is kept exactly as published and can be verified by checksum.

**Expected output.** Archive path and size (≈ a few MB), the number of files in the manifest (≈ 2,000+), and `[OK] 2.2 passed`.

In [ ]:
# ---- 2.2 Preserve original dataset ----
import tarfile

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def build_manifest(root):
    rows = []
    for p in sorted(Path(root).rglob("*")):
        if p.is_file() and ".git" not in p.relative_to(root).parts:
            rows.append({"path": p.relative_to(root).as_posix(),
                         "size_bytes": p.stat().st_size,
                         "sha256": sha256_file(p)})
    return pd.DataFrame(rows)

tag           = BUGSINPY_VERSION["commit"][:12]
archive_path  = PATHS["raw"] / f"BugsInPy_{tag}.tar.gz"
manifest_path = PATHS["raw"] / f"BugsInPy_{tag}_manifest.csv"
source_path   = PATHS["raw"] / f"BugsInPy_{tag}_source.json"

if archive_path.exists():
    print("Archive already exists — NOT overwritten:", archive_path)
else:
    tmp = PATHS["work"] / archive_path.name          # build locally, then copy (faster than writing to Drive)
    with tarfile.open(tmp, "w:gz") as tar:
        tar.add(BUGSINPY_DIR, arcname="BugsInPy")
    shutil.copy2(tmp, archive_path); tmp.unlink()
    print("Archive created:", archive_path)

manifest_df = build_manifest(BUGSINPY_DIR)
if not manifest_path.exists():
    manifest_df.to_csv(manifest_path, index=False)

archive_sha = sha256_file(archive_path)
if not source_path.exists():
    save_json({**BUGSINPY_VERSION, "archive": archive_path.name, "archive_sha256": archive_sha,
               "n_files": len(manifest_df), "total_bytes": int(manifest_df.size_bytes.sum())}, source_path)

print(f"Archive size   : {archive_path.stat().st_size / 1024**2:.2f} MB")
print(f"Archive SHA-256: {archive_sha}")
print(f"Files in manifest (excluding .git): {len(manifest_df)}  "
      f"({manifest_df.size_bytes.sum() / 1024**2:.2f} MB)")

# ---- Verification: archive is readable and complete; local files match the stored manifest ----
with tarfile.open(archive_path, "r:gz") as tar:
    n_arch = sum(1 for m in tar.getmembers() if m.isfile() and "/.git/" not in m.name)
assert n_arch == len(manifest_df), f"Archive has {n_arch} files, manifest {len(manifest_df)}"
stored = pd.read_csv(manifest_path)
assert stored.equals(manifest_df), "Local clone differs from the stored manifest!"
print("\n[OK] 2.2 passed")

**Interpretation.** The raw data is now preserved in Drive and protected by checksums. If you re-run this cell later, it will **not** overwrite the archive. It will, however, check that the local files still match the manifest saved the first time.

## 2.3 Directory structure

**Explanation.** We display the top-level layout, the list of projects with their number of bug folders, and the full content of one example bug.

**Scientific objective.** Understand the raw format before writing any parser, and detect unexpected files.

**Expected output.**
* Top level: `Dockerfile`, `README.md`, `framework/`, `projects/` …
* A table of projects (≈ 17) with bug counts.
* Any **unexpected entries**. For example, when this notebook was prepared there was a stray Office lock file `projects/pandas/bugs/~$Bugs info.xlsx`. It is not a bug folder and will be ignored.
* The content of `black` bug 1: `project.info`, `bug.info` and the first lines of `bug_patch.txt`.

In [ ]:
# ---- 2.3 Directory structure ----
PROJECTS_DIR = BUGSINPY_DIR / "projects"

print("Top level of the repository:")
for p in sorted(BUGSINPY_DIR.iterdir()):
    if p.name != ".git":
        print("  ", p.name + ("/" if p.is_dir() else ""))

rows, unexpected = [], []
for proj in sorted(PROJECTS_DIR.iterdir(), key=lambda p: p.name.lower()):
    if not proj.is_dir():
        unexpected.append(proj.relative_to(BUGSINPY_DIR).as_posix()); continue
    bug_dirs = [b for b in (proj / "bugs").iterdir() if b.is_dir()] if (proj / "bugs").is_dir() else []
    unexpected += [b.relative_to(BUGSINPY_DIR).as_posix()
                   for b in (proj / "bugs").iterdir() if not b.is_dir()] if (proj / "bugs").is_dir() else []
    rows.append({"project": proj.name, "n_bug_folders": len(bug_dirs),
                 "has_project_info": (proj / "project.info").is_file(),
                 "other_files": ", ".join(sorted(f.name for f in proj.iterdir()
                                                 if f.name not in ("bugs", "project.info")))})
structure_df = pd.DataFrame(rows)
print(f"\nProjects found: {len(structure_df)}")
display(structure_df)
print("Unexpected (non-folder) entries:", unexpected or "none")

# Example bug
ex = PROJECTS_DIR / "black" / "bugs" / "1"
if not ex.is_dir():
    ex = next(b for b in (PROJECTS_DIR / structure_df.project[0] / "bugs").iterdir() if b.is_dir())
print(f"\n=== Example: {ex.relative_to(BUGSINPY_DIR)} ===")
print("Files:", sorted(f.name for f in ex.iterdir()))
print("\n--- project.info ---\n" + (ex.parent.parent / "project.info").read_text().strip())
print("\n--- bug.info ---\n" + (ex / "bug.info").read_text().strip())
print("\n--- bug_patch.txt (first 25 lines) ---")
print("\n".join((ex / "bug_patch.txt").read_text(errors="replace").splitlines()[:25]))

# ---- Verification ----
assert len(structure_df) > 0, "No projects found."
assert structure_df.has_project_info.all(), "Some projects have no project.info"
print("\n[OK] 2.3 passed")

**Interpretation.** The diff example shows the label source we will use in Phase 4: lines starting with `-` exist only in the **buggy** version, and lines starting with `+` exist only in the **fixed** version. Unexpected entries are recorded and skipped; they are not deleted, because the raw data must stay untouched.

## 2.4 Verify accessibility and well-formedness of every bug

**Explanation.** For **every** bug folder we check that the required files exist and can be read, then parse `project.info` and `bug.info` (format: `key="value"`) into a table called the *bug inventory*.

Checks:
* `bug.info`, `bug_patch.txt` and `run_test.sh` exist;
* `buggy_commit_id` and `fixed_commit_id` are present and are valid hexadecimal git hashes;
* the patch is non-empty, readable and modifies at least one `.py` file.

**Scientific objective.** Detect broken or atypical entries *before* they silently corrupt the dataset. Anomalies are reported, not hidden.

**Note — short commit hashes.** When this notebook was prepared, the `pandas` bugs used **abbreviated** buggy commit IDs (7 characters, e.g. `45fee32`) instead of full 40-character hashes. These are still valid git references, and Phase 4 will expand them to full hashes when checking out the code. They are therefore flagged as `short_buggy_sha` (a note, not an error).

**Expected output.** An anomaly table and a summary: number of bugs found, number fully valid, and number with notes. The inventory is saved to `data/interim/bugsinpy_inventory.csv`.

In [ ]:
# ---- 2.4 Accessibility & well-formedness ----
KV = re.compile(r'(\w+)\s*=\s*"([^"]*)"')
HEX = re.compile(r"[0-9a-f]{7,40}")

def parse_info(path):
    return dict(KV.findall(Path(path).read_text(encoding="utf-8", errors="replace")))

records, anomalies = [], []
for proj in sorted(p for p in PROJECTS_DIR.iterdir() if p.is_dir()):
    pinfo = parse_info(proj / "project.info") if (proj / "project.info").is_file() else {}
    for bdir in sorted((b for b in (proj / "bugs").iterdir() if b.is_dir()), key=lambda b: int(b.name)):
        rec = {"project": proj.name, "bug_id": int(bdir.name),
               "sample_bug_key": f"{proj.name}-{bdir.name}",
               "github_url": pinfo.get("github_url")}
        issues = []
        for f in ("bug.info", "bug_patch.txt", "run_test.sh"):
            rec[f"has_{f.split('.')[0]}"] = (bdir / f).is_file()
            if not rec[f"has_{f.split('.')[0]}"]:
                issues.append(f"missing_{f}")
        info = parse_info(bdir / "bug.info") if rec["has_bug"] else {}
        for k in ("buggy_commit_id", "fixed_commit_id", "test_file", "python_version", "pythonpath"):
            rec[k] = info.get(k)
        for k in ("buggy_commit_id", "fixed_commit_id"):
            v = (rec[k] or "").strip()
            if not HEX.fullmatch(v):
                issues.append(f"invalid_{k}")
            elif len(v) < 40:
                issues.append(f"short_{k.split('_')[0]}_sha")
        if rec["has_bug_patch"]:
            try:
                patch = (bdir / "bug_patch.txt").read_text(encoding="utf-8")
            except UnicodeDecodeError:
                patch = (bdir / "bug_patch.txt").read_text(encoding="latin-1"); issues.append("patch_not_utf8")
            files_changed = re.findall(r"^diff --git a/(\S+) b/", patch, flags=re.M)
            rec["patch_lines"] = len(patch.splitlines())
            rec["patch_files_changed"] = len(files_changed)
            rec["patch_py_files_changed"] = sum(f.endswith(".py") for f in files_changed)
            if not patch.strip():                 issues.append("empty_patch")
            elif rec["patch_py_files_changed"] == 0: issues.append("no_py_file_in_patch")
        rec["has_setup_sh"] = (bdir / "setup.sh").is_file()
        rec["has_requirements"] = (bdir / "requirements.txt").is_file()
        rec["issues"] = ";".join(issues)
        records.append(rec)

inventory_df = pd.DataFrame(records)
NOTE_ONLY = {"short_buggy_sha", "short_fixed_sha"}
inventory_df["is_valid"] = inventory_df.issues.apply(
    lambda s: all(i in NOTE_ONLY for i in s.split(";") if i))

issue_counts = Counter(i for s in inventory_df.issues for i in s.split(";") if i)
print("Issue / note counts:", dict(issue_counts) or "none")
print("\nIssues per project:")
display(inventory_df[inventory_df.issues != ""].groupby("project").issues
        .agg(lambda s: dict(Counter(i for x in s for i in x.split(";")))).to_frame())

inv_path = PATHS["interim"] / "bugsinpy_inventory.csv"
inventory_df.to_csv(inv_path, index=False)
print(f"\nBugs found          : {len(inventory_df)}")
print(f"Valid (incl. notes) : {inventory_df.is_valid.sum()}")
print(f"With blocking issues: {(~inventory_df.is_valid).sum()}")
print("Saved:", inv_path)
display(inventory_df.head())

# ---- Verification ----
assert inventory_df.sample_bug_key.is_unique, "Duplicate bug keys!"
assert inventory_df.has_bug.all() and inventory_df.has_bug_patch.all(), "Some bugs miss bug.info / bug_patch.txt"
assert inventory_df.is_valid.mean() > 0.95, "More than 5% of bugs are malformed — investigate before continuing."
print("\n[OK] 2.4 passed")

**Interpretation.** `sample_bug_key` (e.g. `black-1`) uniquely identifies each bug instance and will be carried through every later phase, which is necessary to prevent bug-instance leakage in Phase 6. Bugs with blocking issues are **not deleted**; they will be excluded in Phase 4, and the exclusion will be reported. `patch_py_files_changed` already shows that some fixes touch several files, which matters for the labeling strategy.

## 2.5 Record the dataset version and Phase 2 summary

**Explanation.** The exact dataset version (commit, date, archive checksum) is written into `CONFIG` and the config file of this run is updated.

**Expected output.** The updated `dataset` section of `CONFIG` and a summary table with `PASS` on every row.

In [ ]:
# ---- 2.5 Record dataset version & summary ----
CONFIG["dataset"].update({
    "commit": BUGSINPY_VERSION["commit"],
    "commit_date": BUGSINPY_VERSION["commit_date"],
    "downloaded_at": BUGSINPY_VERSION["downloaded_at"],
    "raw_archive": str(archive_path),
    "raw_archive_sha256": archive_sha,
    "manifest": str(manifest_path),
    "inventory": str(inv_path),
    "n_projects": int(inventory_df.project.nunique()),
    "n_bugs": int(len(inventory_df)),
})
cfg_path = save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
print(json.dumps(CONFIG["dataset"], indent=2))

# integrity re-check: originals untouched
still_same = build_manifest(BUGSINPY_DIR).equals(pd.read_csv(manifest_path))
checks2 = {
    "Repository cloned (commit recorded)":  "PASS" if CONFIG["dataset"]["commit"] else "FAIL",
    "Raw archive preserved on storage":     "PASS" if archive_path.exists() else "FAIL",
    "Manifest + checksums saved":           "PASS" if manifest_path.exists() else "FAIL",
    "Original files unmodified":            "PASS" if still_same else "FAIL",
    "All bug folders readable":             "PASS" if inventory_df.has_bug.all() else "FAIL",
    f"Valid bugs: {inventory_df.is_valid.sum()}/{len(inventory_df)}": "PASS",
    "Config updated with dataset version":  "PASS" if json.loads(cfg_path.read_text())["dataset"]["commit"] else "FAIL",
}
print()
print(pd.DataFrame({"check": list(checks2), "status": list(checks2.values())}).to_string(index=False))
assert "FAIL" not in checks2.values()
print(f"\n[OK] PHASE 2 COMPLETE — dataset version {CONFIG['dataset']['commit'][:12]} "
      f"({CONFIG['dataset']['n_projects']} projects, {CONFIG['dataset']['n_bugs']} bugs).")

**Interpretation.** The dataset is downloaded, versioned, preserved and verified. Please report back:
1. The output of **2.1** (commit hash and date),
2. The project table of **2.3** and any unexpected entries,
3. The issue counts and summary of **2.4**,
4. The summary table of **2.5**.

➡️ **PHASE 3 (Dataset exploration) will be added after these results are confirmed.**